In [ ]:
from collections import Counter

#source, hyp, ref
def GLUE(S,H,R):
    pass
    
    
#Score=∣H∩R∣−∣H∩S*∣
def score(S,H,R):
    n = 4
    Sg = get_all_n_grams(S,n)
    Hg = get_all_n_grams(H,n)
    Rg = get_all_n_grams(R,n)

    penalty = len(set(Hg) & (set(Sg) - set(Rg)))
    
    print(penalty)
    
    score = len(set(Hg) & set(Rg)) - penalty
    
    return score / len(Hg) if len(Hg) > 0 else 1


def n_grams(X, n):
    if type(X) == str:
        X = X.split()
    return [tuple(X[i:i+n]) for i in range(len(X)-n+1)]

def get_all_n_grams(X, max_n):
    buffer = [] #index 1: uni-grams, index 2: bi-grams etc. 
    for n in range(1, max_n+1):
        buffer.append(n_grams(X,n))
        
    flattend = [] #store all grams into 1 list instead of sublists by unrolling the buffer. 
    for subset in buffer:
        flattend += subset
        
    return flattend


#examples

#get_all_n_grams("lotje leerde linde lopen op de lange linde laan", 4)

score("A B","A B","A B")





0


1.0

In [ ]:
import math
from collections import Counter #hashmap like set that keeps track over unique key feqs

def n_grams(seq, n):
    if type(seq) == str:
        seq = seq.split()
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

def GLEU(S, H, R, max_n = 4):
    if(len(H) < 1): 
        return "HPY empty"
    
    #tokenize on wite space
    tok_H = H.split() if isinstance(H, str) else H
    tok_R = R.split() if isinstance(R, str) else R
    #brevity penalty that pushed the model for producing shorter hypotheses 
    BP = 1 if len(tok_H) >= len(tok_R) else math.exp(1 - (len(tok_R)/len(tok_H)))
    
    precision_at_n = []
    
    for n in range(1, max_n +1): 
        Sg = n_grams(S, n)
        Hg = n_grams(H, n)
        Rg = n_grams(R, n)
        
        #convert n-gram collections to dict with feq counts as values
        CS, CH, CR = Counter(Sg), Counter(Hg), Counter(Rg)
        
        desired_overlap = CH & CR
        source_overlap = CH & CS
        undesired_overlap = source_overlap - desired_overlap  
        #score clipping the diff between correctly copied n-grams and incorrectly copied n-grams between 0 and +inf,
        #divided by the n-gram count of hyp. 
        precision_at_n.append(max(0, sum(desired_overlap.values()) - sum(undesired_overlap.values())) / sum(CH.values())) 
    
    geometric_mean = math.exp(sum(math.log(p) for p in precision_at_n)/max_n)
    
    return geometric_mean * BP

In [ ]:
import math
from collections import Counter #hashmap like set that keeps track over unique key feqs

def n_grams(seq, n):
    if type(seq) == str:
        seq = seq.split()
    return [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]

def GLEU(S, H, R, max_n = 4, _lambda = 1):    
    #tokenize on wite space
    tok_H = H.split() if isinstance(H, str) else H
    tok_R = R.split() if isinstance(R, str) else R
    #brevity penalty that pushed the model for producing shorter hypotheses 
    BP = 1 if len(tok_H) >= len(tok_R) else math.exp(1 - (len(tok_R)/len(tok_H)))
    
    precision_at_n = []
    
    for n in range(1, max_n +1): 
        Sg = n_grams(S, n)
        Hg = n_grams(H, n)
        Rg = n_grams(R, n)
        
        # convert n-gram collections to dict with freq counts as values
        CS, CH, CR = Counter(Sg), Counter(Hg), Counter(Rg)
                
        # n-grams present in the candidate that overlap with the reference but not the source (R \ S)
        count_R_S = CH & (CR - CS)
        
        # n-grams in the candidate that are in the source but not the reference (S \ R)
        count_S_R = CH & (CS - CR)
        
        # base overlap between candidate and reference
        count_R = CH & CR
        
        #n-gram freqs
        val_R_S = sum(count_R_S.values())
        val_S_R = sum(count_S_R.values())
        val_R = sum(count_R.values())
        
        denominator = sum(CH.values())
        
        if denominator > 0:
            # Apply Equation 1: (Base Overlap + Reward - Penalty) / Denominator
            numerator = val_R + val_R_S - (_lambda * val_S_R)
            
            # Clip at near zero to prevent negative precision, use 1e-7 to prevent devide by zero error. 
            precision = max(1e-7, numerator) / denominator
            precision_at_n.append(precision)
        else:
            precision_at_n.append(1e-7)
    
    geometric_mean = math.exp(sum(math.log(p) for p in precision_at_n)/max_n)
    
    return geometric_mean * BP